# Pipeline 5 — Safehouse Incident Forecast (Next Month)

**Created:** 2026-04-06T17:23:54.995881Z

This notebook follows **CRISP-DM** while also satisfying the IS455 rubric sections:
- Problem Framing
- Data Acquisition, Preparation & Exploration
- Modeling & Feature Selection
- Evaluation & Interpretation
- Causal and Relationship Analysis
- Deployment Notes


## CRISP-DM Overview

### 1) Business Understanding
Goal: forecast next-month incident risk to proactively allocate resources and staffing.

### 2) Data Understanding
Use safehouse_monthly_metrics time series per safehouse.

### 3) Data Preparation
Create lag/rolling features and time-based split.

### 4) Modeling
Predictive: Random Forest Regressor. Explanatory: Linear Regression.

### 5) Evaluation
Evaluate forecast error and discuss operational consequences of underestimating risk.

### 6) Deployment
Export predicted incident forecast and risk band per safehouse; import into `/api/ml/import`.


## 1) Problem Framing (Rubric)

State:
- the business question,
- who cares,
- why it matters,
- predictive vs explanatory goals.

We build **two models**:
- Predictive (optimize out-of-sample performance)
- Explanatory (interpretability / relationship analysis)


## 2) Data Acquisition, Preparation & Exploration (Rubric)

Rules to avoid leakage:
- Define an **as-of date** (cutoff).
- Build features using only data **on or before** the cutoff.
- Create labels using only data **after** the cutoff in a defined horizon.


In [ ]:
import json
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor


REPO_ROOT = Path("..").resolve()
RAW_DIR_A = (REPO_ROOT / "data" / "raw" / "lighthouse_csv_v7").resolve()
RAW_DIR_B = (REPO_ROOT / "data" / "raw").resolve()
DATA_DIR = RAW_DIR_A if RAW_DIR_A.exists() else RAW_DIR_B

OUT_DIR = (REPO_ROOT / "output" / "ml-predictions").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data dir:", DATA_DIR)
print("Out dir:", OUT_DIR)


def require_csv(stem: str) -> pd.DataFrame:
    path = DATA_DIR / f"{stem}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}.")
    return pd.read_csv(path, encoding="utf-8-sig")


def to_date(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce").dt.date


def to_dt(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce", utc=True)


def eval_classification(y_true, y_pred, y_proba=None):
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    out = {"accuracy": float(acc), "precision": float(pr), "recall": float(rc), "f1": float(f1)}
    if y_proba is not None:
        try:
            out["roc_auc"] = float(roc_auc_score(y_true, y_proba))
        except Exception:
            pass
    return out


def eval_regression(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred, squared=False)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def export_predictions_json(prediction_type: str, entity_type: str, df_out: pd.DataFrame, id_col: str, score_col: str, label_col: str | None = None):
    out_path = OUT_DIR / f"{prediction_type}.json"
    rows = []
    for _, r in df_out.iterrows():
        rows.append(
            {
                "predictionType": prediction_type,
                "entityType": entity_type,
                "entityId": int(r[id_col]),
                "score": float(r[score_col]),
                "label": None if label_col is None else (None if pd.isna(r[label_col]) else str(r[label_col])),
                "payloadJson": json.dumps({k: v for k, v in r.items() if k not in {id_col, score_col, label_col}}, default=str),
            }
        )
    out_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
    print("Wrote:", out_path, "rows=", len(rows))


In [ ]:
metrics = require_csv("safehouse_monthly_metrics")
metrics["month_start"] = pd.to_datetime(metrics["month_start"], errors="coerce")
metrics = metrics.dropna(subset=["month_start"]).sort_values(["safehouse_id","month_start"]).copy()

for c in ["active_residents","avg_education_progress","avg_health_score","process_recording_count","home_visitation_count","incident_count"]:
    metrics[c] = pd.to_numeric(metrics[c], errors="coerce").fillna(0)

# Lag features
g = metrics.groupby("safehouse_id")
metrics["incident_lag1"] = g["incident_count"].shift(1)
metrics["incident_lag2"] = g["incident_count"].shift(2)
metrics["active_lag1"] = g["active_residents"].shift(1)
metrics["rolling_incident_3m"] = g["incident_count"].rolling(3).mean().reset_index(level=0, drop=True)

# Label: next month incident_count
metrics["incident_next"] = g["incident_count"].shift(-1)

df = metrics.dropna(subset=["incident_next","incident_lag1","active_lag1"]).copy()

features = ["active_residents","avg_education_progress","avg_health_score","process_recording_count","home_visitation_count","incident_count",
            "incident_lag1","incident_lag2","active_lag1","rolling_incident_3m"]

cutoff = df["month_start"].quantile(0.8)
train = df[df["month_start"] <= cutoff].copy()
test = df[df["month_start"] > cutoff].copy()

X_train, y_train = train[features], train["incident_next"]
X_test, y_test = test[features], test["incident_next"]
print("Train:", len(train), "Test:", len(test))


In [ ]:
# Predictive model
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
print("Predictive (RF):", eval_regression(y_test, pred))

# Explanatory model
lin = LinearRegression()
lin.fit(X_train, y_train)
pred2 = lin.predict(X_test)
print("Explanatory (Linear):", eval_regression(y_test, pred2))


In [ ]:
# Score next-month incident risk for each safehouse (latest month per safehouse)
latest = metrics.groupby("safehouse_id").tail(1).copy()
latest = latest.dropna(subset=["incident_lag1","active_lag1"])
latest["predicted_incidents_next_month"] = rf.predict(latest[features])
latest["risk_band"] = pd.qcut(latest["predicted_incidents_next_month"].rank(method="first"), q=4, labels=["Low","Medium","High","Very High"])

export_predictions_json(
    prediction_type="safehouse_incident_next_month",
    entity_type="Safehouse",
    df_out=latest[["safehouse_id","predicted_incidents_next_month","risk_band","active_residents","incident_count","avg_health_score","avg_education_progress"]].rename(columns={"safehouse_id":"safehouse_id","predicted_incidents_next_month":"score"}),
    id_col="safehouse_id",
    score_col="score",
    label_col="risk_band"
)


## 3) Modeling & Feature Selection (Rubric)

- Predictive model: tree/ensemble
- Explanatory model: linear/logistic regression


## 4) Evaluation & Interpretation (Rubric)

Interpret in business terms, and discuss real-world costs of errors.

## 5) Causal and Relationship Analysis (Rubric)

Discuss relationships, confounding risks, and where correlation ≠ causation.


## 6) Deployment Notes (Rubric)

Export predictions to JSON and import into the deployed app:
- `POST /api/ml/import?replace=true` (admin-only)
- View in `/app/ml` (Staff Portal → ML Insights)
